# Electrical Grid Stability Classification
## Notebook 02: Comparative Model Analysis & Kernel Diagnostics

This notebook provides a deep-dive evaluation of 5 benchmarked classification architectures:
1. **Support Vector Machine — Linear Kernel**
2. **Support Vector Machine — Polynomial Kernel**
3. **Support Vector Machine — Radial Basis Function (RBF) Kernel**
4. **K-Nearest Neighbors (KNN)**
5. **Decision Tree Classifier**

All evaluation metrics and comparisons are loaded directly from generated pipeline artifacts.

In [ ]:
import sys
import json
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.config import MODEL_COMPARISON_PATH, METRICS_PATH, FEATURE_IMPORTANCE_PATH, BEST_MODEL_PATH
from src.prediction import load_model
from src.visualization import PCA_DISCLAIMER, compute_pca_projection

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

### 1. Benchmark Model Comparison Summary
Examining the actual cross-validation and holdout test performance across all 5 candidate models.

In [ ]:
df_comp = pd.read_csv(MODEL_COMPARISON_PATH)
print("Model Comparison Leaderboard (Ranked by CV F1 Score):")
df_comp[["model", "cv_f1_mean", "cv_accuracy_mean", "test_f1", "test_accuracy", "training_time_sec"]]

### 2. SVM Kernel Comparison: Linear vs Polynomial vs RBF
Comparing how different kernel transformations capture nonlinear dynamical phase boundaries.

In [ ]:
svm_df = df_comp[df_comp["model"].str.contains("SVM")].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy & F1
svm_melted = pd.melt(svm_df, id_vars=["model"], value_vars=["test_accuracy", "test_f1"], var_name="Metric", value_name="Score")
sns.barplot(data=svm_melted, x="model", y="Score", hue="Metric", palette=["#1f77b4", "#ff7f0e"], ax=ax1)
ax1.set_ylim(0.75, 1.0)
ax1.set_title("SVM Kernels: Holdout Accuracy vs F1")
ax1.set_xlabel("")

# Training Time
sns.barplot(data=svm_df, x="model", y="training_time_sec", palette="Blues_d", ax=ax2)
ax2.set_title("SVM Kernels: Training Compute Time (seconds)")
ax2.set_xlabel("")
ax2.set_ylabel("Seconds")

plt.tight_layout()
plt.show()

### 3. Model Explainability: Permutation Feature Importance
Permutation importance measures the drop in F1 performance when each feature is randomly shuffled on the test set.

In [ ]:
df_imp = pd.read_csv(FEATURE_IMPORTANCE_PATH)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=df_imp.sort_values(by="importance_mean", ascending=False),
    x="importance_mean",
    y="feature",
    palette="crest"
)
plt.title("Permutation Feature Importance (Decrease in Holdout F1)", fontsize=13)
plt.xlabel("Mean Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("Important Disclaimer: Feature importance reflects model behavior and does not establish physical causality.")

### 4. 2D PCA Decision Space Projection
Visualizing the parameter space via principal component analysis.

In [ ]:
from src.data_loader import load_raw_data
from src.validation import validate_raw_dataset

df_raw = load_raw_data()
X, y, _ = validate_raw_dataset(df_raw)
pca_df, pca = compute_pca_projection(X, y)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Label", palette={"Stable": "#2ca02c", "Unstable": "#d62728"}, alpha=0.5)
plt.title(f"2D PCA Projection (Explained Variance: {pca.explained_variance_ratio_[0]*100:.1f}% + {pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.figtext(0.5, -0.02, PCA_DISCLAIMER, wrap=True, horizontalalignment='center', fontsize=9, style='italic')
plt.tight_layout()
plt.show()